# ISM Project Shafe
## Leakage-Resistant Insider Threat Detection on CERT r4.2

**Full Workflow, Code, Evaluation, Uncertainty, Explainability and Operational Decision System**

| Item | Value |
|---|---|
| Dataset | CERT Insider Threat Dataset r4.2 |
| Analytical Unit | user × day |
| Final Production Model | `lgbm-graph-v1` (LightGBM 4.6.0, 12 features) |
| Model Development Status | **STOPPED** |
| Final System Status | **COMPLETE** |
| Project Seed | 42 |
| Alert Threshold | 0.9186015432508062 (frozen) |

---

## Executive Overview

**Goal**: Detect risky user-days while producing:
- Risk score (ML prediction)
- Uncertainty/confidence (conformal prediction)
- Explanations (SHAP contributions)
- Graph/trust diagnostics
- Recommended action

**Final Architecture**:
```
CERT r4.2 logs
→ validation/preprocessing
→ leakage-safe user-day aggregation
→ 8 behavioral features (frozen)
→ 4 graph/trust features (frozen)
→ LightGBM lgbm-graph-v1 (frozen)
→ frozen risk policy (threshold 0.9186)
→ conformal uncertainty overlay
→ SHAP explanations
→ trust diagnostics
→ deterministic decision engine
→ dashboard-ready output (501K rows × 21 columns)
```

**Adaptive Risk is NOT in the production path** — it was rejected after failing independent validation.

## Project Structure

```
ISM_Project_Shafe/
├── src/                    # Modular source code
│   ├── config.py           # Feature registry, splits, constants
│   ├── data/               # Validation, labels, aggregation
│   ├── experiments/        # Phase 6-20 experiment modules
│   ├── graph/              # Graph feature construction
│   ├── models/             # LightGBM baseline
│   ├── evaluation/         # Metrics, threshold
│   └── preprocessing/      # Alignment, splits
├── configs/phase20/        # Frozen decision policy, output schema
├── reports/                # Phase reports + artifacts
├── tests/                  # 54+ tests (Phase 20)
├── knowledge/              # Project knowledge (Markdown)
└── teacher_demo/           # THIS NOTEBOOK
```

*The notebook is the demonstration layer. The actual implementation remains modular and reproducible.*

In [ ]:
# ---- DEMO SETUP ----
DEMO_MODE = True

import sys, os, json, warnings, hashlib
warnings.filterwarnings('ignore')

# Detect environment
IN_KAGGLE = 'kaggle' in os.environ.get('KAGGLE_RUNNING_CLASS', '').lower() or \
            os.path.exists('/kaggle/input')

# Project root - search multiple locations
def find_project_root():
    candidates = []
    if IN_KAGGLE:
        candidates = ['/kaggle/working', '/kaggle/input']
    else:
        candidates = [
            os.path.abspath('.'),
            os.path.abspath('..'),
            os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else '.',
        ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'src')) or os.path.exists(os.path.join(c, 'reports')):
            return c
    return os.path.abspath('.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, PROJECT_ROOT)

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Local'}")
print(f"Project root: {PROJECT_ROOT}")
print(f"DEMO_MODE: {DEMO_MODE}")

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
print('matplotlib backend:', matplotlib.get_backend())


## Reproducibility / Environment

In [ ]:
import platform, sys

env_info = {
    'Python': sys.version.split()[0],
    'Platform': platform.platform(),
    'Project Seed': 42,
    'Frozen Model': 'lgbm-graph-v1',
    'LightGBM (kernel)': '4.6.0',
    'Frozen Threshold': 0.9186015432508062,
    'Conformal alpha': 0.05,
    'Conformal t0': 0.4634739481800199,
    'Conformal t1': 0.0046536002164601275,
}

print("=" * 60)
print("FROZEN ENVIRONMENT")
print("=" * 60)
for k, v in env_info.items():
    print(f"  {k:25s} {v}")
print()

# Try importing key packages
try:
    import lightgbm as lgb
    print(f"  LightGBM (actual):       {lgb.__version__}")
except ImportError:
    print("  LightGBM:                not installed (frozen artifact used)")

try:
    import pandas as pd
    print(f"  pandas:                  {pd.__version__}")
except ImportError:
    print("  pandas:                  not installed")

try:
    import numpy as np
    print(f"  numpy:                   {np.__version__}")
except ImportError:
    print("  numpy:                   not installed")

## Dataset Overview

**CERT r4.2** — Carnegie Mellon insider threat simulation dataset.

| Source | Description |
|---|---|
| logon.csv | User login/logout events |
| device.csv | USB device connections |
| file.csv | File access events |
| http.csv | HTTP activity (14.5 GB) |
| ldap.csv | LDAP snapshots (monthly) |
| answers/insiders.csv | Ground truth labels (70 incidents) |

**Population**:
- 1,000 users × 501 days = **501,000 user-day rows**
- 1,892 malicious rows (70 users, 0.38% prevalence)
- Date range: 2010-01-02 to 2011-05-17

In [ ]:
import json

# Load dataset report if available
try:
    with open(os.path.join(PROJECT_ROOT, 'reports/artifacts/dataset_report.json')) as f:
        ds_report = json.load(f)
    print("FROZEN VALIDATED RESULT — dataset_report.json")
    print(f"  Users:            {ds_report.get('n_users', 'N/A')}")
    print(f"  Date range:       {ds_report.get('date_min', 'N/A')} to {ds_report.get('date_max', 'N/A')}")
    print(f"  Malicious rows:   {ds_report.get('n_malicious_rows', 'N/A')}")
    print(f"  Malicious users:  {ds_report.get('n_malicious_users', 'N/A')}")
except FileNotFoundError:
    print("Using authoritative values from master report")
    print("  Users:            1,000")
    print("  User-days:        501,000")
    print("  Malicious rows:   1,892 (0.38%)")
    print("  Malicious users:  70")
    print("  Date range:       2010-01-02 to 2011-05-17")

## Class Imbalance

With only **0.38% prevalence** (1,892 / 501,000), accuracy is meaningless — a model predicting all-benign achieves 99.62% accuracy.

**Important metrics**:
- **PR-AUC** (Precision-Recall AUC): discriminative power under imbalance
- **Precision / Recall / F1**: operating-point tradeoffs
- **MCC** (Matthews Correlation Coefficient): balanced measure
- **Alert volume**: operational feasibility

In [ ]:
# Validation checks on the final decision table

import pandas as pd

# Parquet hash (authoritative)
EXPECTED_MD5 = '6476f791d1cc9f21327a94e1373f9e09'
EXPECTED_SHA256 = 'f03ffaf7d5512405696103286ed726246b2280fbcdbb3038494c5e169d70d9b3'

# Search multiple paths for the parquet
parquet_search = [
      '/kaggle/working/final_user_day_decisions.parquet',
      '/kaggle/working/teacher_demo_final/demo_artifacts/final_user_day_decisions.parquet',
    os.path.join(PROJECT_ROOT, 'teacher_demo', 'demo_artifacts', 'final_user_day_decisions.parquet'),
    os.path.join(PROJECT_ROOT, 'reports', 'artifacts', 'phase20', 'final_user_day_decisions.parquet'),
    '/kaggle/working/reports/artifacts/phase20/final_user_day_decisions.parquet',
    '/kaggle/working/teacher_demo/demo_artifacts/final_user_day_decisions.parquet',
]

df = None
DATA_SOURCE = 'NONE'
FULL_TABLE_HASH_VERIFIED = 'NO'

for pp in parquet_search:
    if os.path.exists(pp):
        # Verify hash before loading
        try:
            with open(pp, 'rb') as f:
                file_bytes = f.read()
            file_md5 = hashlib.md5(file_bytes).hexdigest()
            file_sha256 = hashlib.sha256(file_bytes).hexdigest()
            
            if file_md5.lower() == EXPECTED_MD5.lower():
                FULL_TABLE_HASH_VERIFIED = 'YES'
                df = pd.read_parquet(pp)
                DATA_SOURCE = 'FULL_PRODUCTION_TABLE'
                print(f'DATA_SOURCE = {DATA_SOURCE}')
                print(f'Loaded from: {pp}')
                print(f'Hash verified: MD5={file_md5[:16]}... SHA256={file_sha256[:16]}...')
                break
            else:
                print(f'Hash mismatch at {pp}: got {file_md5}, expected {EXPECTED_MD5}')
        except Exception as e:
            print(f'Failed to load {pp}: {e}')

if df is not None:
    TABLE_LOADED = True
    print()
    print('Phase 20 Decision Table Validation')
    print('=' * 55)
    print(f'  Shape:                  {df.shape}')
    print(f'  Columns ({len(df.columns)}):     {list(df.columns)}')
    uid_col = next((c for c in ['user_id','user'] if c in df.columns), 'user_id')
    date_col = next((c for c in ['date','day'] if c in df.columns), 'date')
    risk_col = next((c for c in ['ml_risk_score','ml_risk'] if c in df.columns), 'ml_risk_score')
    print(f'  Unique users:           {df[uid_col].nunique()}')
    print(f'  Date range:             {df[date_col].min()} to {df[date_col].max()}')
    print(f'  Missing values:         {df.isnull().sum().sum()}')
    print(f'  Duplicate rows:         {df.duplicated().sum()}')
    print(f'  ml_risk range:          [{df[risk_col].min():.6f}, {df[risk_col].max():.6f}]')
    print(f'  Risk levels:            {dict(df["risk_level"].value_counts())}')
    print(f'  Full table hash:        {FULL_TABLE_HASH_VERIFIED}')
    print()
    # Validate shape
    assert df.shape == (501000, 21), f'Unexpected shape: {df.shape}'
    print(f'  Shape validation: 501,000 x 21 = PASS')
    print(f'  {df.shape[0]:,} rows x {df.shape[1]} columns')
else:
    TABLE_LOADED = False
    DATA_SOURCE = 'FAIL'
    print(f'DATA_SOURCE = FAIL')
    print('FATAL: Could not load authoritative parquet from any path.')
    print('Searched:')
    for pp in parquet_search:
        print(f'  {pp} -> {"EXISTS" if os.path.exists(pp) else "not found"}')
    print('Cannot proceed without the full production table.')

## Data Validation

**LIVE DEMONSTRATION** — validating the user-day table structure.

In [ ]:
if TABLE_LOADED:
    print('Sample rows from the Phase 20 decision table:')
    print('(These are FROZEN VALIDATED RESULTS from the full 501K table)')
    uid_col = next((c for c in ['user_id','user'] if c in df.columns), 'user_id')
    date_col = next((c for c in ['date','day'] if c in df.columns), 'date')
    risk_col = next((c for c in ['ml_risk_score','ml_risk'] if c in df.columns), 'ml_risk_score')
    conf_set_col = next((c for c in ['conformal_prediction_set','conformal_set'] if c in df.columns), 'conformal_prediction_set')
    display_cols = [uid_col, date_col, risk_col, 'risk_level', 'confidence', conf_set_col, 'recommended_action']
    print(df[display_cols].head(10).to_string(index=False))
else:
    print('FAIL: Decision table not loaded. Cannot display samples.')


## User-Day Aggregation

The analytical unit is **user × day**: one row = one user on one day.

Why?
- Insider threats are daily behavioral anomalies
- Prevents label leakage from adjacent days
- Enables chronological splitting
- Matches analyst workflow (daily review)

The aggregation pipeline:
1. Merge all log sources by (user, day)
2. Compute behavioral counts per user-day
3. Apply 28-day strictly-past window for unusual activity detection
4. Merge with graph features (also strictly-past)
5. Align with ground truth labels

## Behavioral Features (8, frozen)

| # | Feature | Definition | Source | Prediction-Time Available? |
|---|---|---|---|---|
| 1 | `login_count` | Daily login events | logon.csv | ✅ Yes |
| 2 | `after_hours_login_count` | Logins outside 8am–6pm | logon.csv | ✅ Yes |
| 3 | `usb_connection_count` | USB device connections | device.csv | ✅ Yes |
| 4 | `file_access_count` | File access events | file.csv | ✅ Yes |
| 5 | `sensitive_file_access_count` | Magic-byte detected files (OLE2/PDF/ZIP) | file.csv | ✅ Yes |
| 6 | `http_activity_count` | HTTP activity events | http.csv | ✅ Yes |
| 7 | `unique_device_count` | Distinct devices used | device.csv | ✅ Yes |
| 8 | `unusual_access_count` | Logons on unused PCs (28-day strictly-past) | logon.csv | ✅ Yes |

All features are **day-local** except `unusual_access_count` which uses a 28-day strictly-past window.

## Graph / Trust Features (4, frozen)

Graph features capture relational patterns across the organizational network.

| # | Feature | Definition | Why Useful? |
|---|---|---|---|
| 1 | `device_consistency_score` | Jaccard similarity: today's PCs vs historical PCs | Detects device-hopping behavior |
| 2 | `rare_device_usage_count` | Count of rarely-used devices | Flags unusual device access |
| 3 | `file_type_consistency_score` | Jaccard similarity: today's file types vs historical | Detects file-type anomalies |
| 4 | `rare_file_type_access_count` | Count of rarely-accessed file types | Flags unusual file access |

**Key finding**: Graph-only is weak (AUC-PR 0.014), but combined with behavioral features, it improves the detector (AUC-PR +74.9% relative).

**Rejected**: `department_file_type_mismatch_count` (zero gain in 5/5 seeds, near-constant).

## Leakage-Safe Splits

**Chronological splitting** — no random splits, no future information.

| Split | Period | Rows | Positives | Purpose |
|---|---|---|---|---|
| TRAIN | ≤ 2011-01-31 | 395,000 | 1,539 | Model fitting |
| CALIBRATION | 2011-02-01 .. 2011-03-31 | 59,000 | 323 | Threshold selection, conformal fit |
| TEST | ≥ 2011-04-01 | 47,000 | 30 | Final evaluation (once) |

**Separate Phase 19 protocol**: user-disjoint DEV (588 users) / CONFIRM (196 users), overlap = 0.

In [ ]:
# Split verification
splits = {
    'TRAIN': {'period': 'through 2011-01-31', 'rows': 395000, 'positives': 1539},
    'CALIBRATION': {'period': '2011-02-01 to 2011-03-31', 'rows': 59000, 'positives': 323},
    'TEST': {'period': 'from 2011-04-01', 'rows': 47000, 'positives': 30},
}

print("FROZEN VALIDATED RESULT — Split Protocol")
print("=" * 55)
for name, info in splits.items():
    print(f"  {name:15s} {info['period']:30s} {info['rows']:>8,} rows  {info['positives']:>5} pos")

print(f"  {'':15s} {'':30s} {'─' * 8}  {'─' * 5}")
print(f"  {'TOTAL':15s} {'':30s} {sum(s['rows'] for s in splits.values()):>8,}  {sum(s['positives'] for s in splits.values()):>5}")

print()
print("  ✅ Train/Cal overlap = 0")
print("  ✅ Train/Test overlap = 0")
print("  ✅ Cal/Test overlap = 0")
print("  ✅ Chronological order preserved")
print("  ✅ No random splitting")

## Model Development History

The project evolved through 20 phases of controlled experimentation:

```
Phase 1-2:  Baseline (6 features) → AUC-ROC 0.914
Phase 3:    +2 behavioral features → AUC-PR +34%
Phase 4:    Ablation + FREEZE lgbm-baseline-v2 (8 features)
Phase 5:    Graph feature construction (5 features)
Phase 6:    Graph regression → graph earns its place
Phase 7:    FREEZE lgbm-graph-v1 (12 features)
Phase 8:    Adaptive risk → REJECTED
Phase 9:    Alert policy FROZEN (threshold 0.9186)
Phase 10:   Conformal overlay ACCEPTED
Phase 11:   Explainability layer ACCEPTED
Phase 12:   Operational envelope → no policy beats frozen P0
Phase 13:   Temporal stability → tail-window risk documented
Phase 14:   User holdout → 13/15 unseen users detected
Phase 15:   Degradation diagnosis → calibration inflation + ranking failures
Phase 16:   Final packaging & verification
Phase 17:   Calibration transfer → RESEARCH-ONLY, mechanical FAIL
Phase 18:   Adaptive Risk v2 → FAIL
Phase 19:   CONFIRM evaluation → FAIL (delta ROC-AUC -0.0281)
Phase 20:   Full integration → 501K decisions materialized
```

In [ ]:
# Conformal results
print('FROZEN VALIDATED RESULT - Conformal Coverage (alpha=0.05)')
print('=' * 60)
print(f'  Method:               Mondrian split conformal')
print(f'  Alpha:                0.05 (95% coverage target)')
print(f'  t0 (negative):        0.4634739481800199')
print(f'  t1 (positive):        0.0046536002164601275')
print(f'  n1 (CAL positives):   323')
print(f'  n0 (CAL negatives):   58,677')
print()
print('  TEST coverage:')
print(f'    Positive coverage:  1.000 (30/30)')
print(f'    Negative coverage:  0.988')
print(f'    Marginal coverage:  0.988')
print(f'    Wilson 90% LB (pos): 0.917')
print()
print('  Prediction set interpretation:')
print('    {0}    -> prediction set contains label 0 only (BENIGN)')
print('    {1}    -> prediction set contains label 1 only (MALICIOUS)')
print('    {0,1}  -> both labels remain plausible / ambiguous')
print('    {}     -> empty set (rare, data issue)')
print()
print('  WARNING: The conformal procedure targets approximately 95%')
print('  repeated-sample coverage under its exchangeability assumptions.')
print('  It does NOT mean that every individual prediction has a 95%')
print('  probability of being correct.')
print()
print('  Monitor band (t0 <= score < alert_threshold):')
print('    CAL:   765 rows, precision 0.061 (~11x prevalence)')
print('    TEST:  532 rows, precision 0.015 (~25x prevalence)')
print('    -> Useful watchlist for analyst review')

In [ ]:
import pandas as pd
import numpy as np

# Ablation data from authoritative phase artifacts
ablation = pd.DataFrame({
    'Model': ['A: Behavioral Only\n(lgbm-baseline-v2)', 
              'B: Graph Only\n(Phase 6 Arm B)', 
              'C: Behavioral + Graph\n(lgbm-graph-v1)'],
    'Features': [8, 5, 12],
    'ROC-AUC': [0.93534, 0.78635, 0.93916],
    'PR-AUC': [0.15304, 0.01431, 0.26778],
    'F1': [0.346, 0.054, 0.354],
    'MCC': [0.350, 0.105, 0.365],
    'Alerts': [22, 417, 49],
    'Precision': [0.409, 0.029, 0.286],
    'Recall': [0.300, 0.400, 0.467],
})

print("FROZEN VALIDATED RESULT — Ablation Study")
print("=" * 90)
print(ablation.to_string(index=False))
print()
print("Key findings:")
print("  • A → C: Graph features add +74.9% PR-AUC relative improvement")
print("  • B alone: Graph-only is weak (PR-AUC 0.014, stump at 3 iterations)")
print("  • C dominates all models by PR-AUC, P@10, R@50")

In [ ]:
# Decision policy - load from frozen config
import yaml

policy_path = os.path.join(PROJECT_ROOT, 'configs', 'phase20', 'decision_policy.yaml')
try:
    with open(policy_path) as f:
        policy_config = yaml.safe_load(f)
    print('FROZEN DECISION POLICY - loaded from configs/phase20/decision_policy.yaml')
    print('=' * 100)
    for level in policy_config.get('risk_levels', []):
        name = level['name']
        cond = level.get('condition', 'N/A')
        action = level.get('action', 'N/A')
        urgency = level.get('urgency', 'N/A')
        print(f'  {name:12s} | {cond:40s} | {action:35s} | {urgency}')
    print()
    # Show alert_threshold specifically
    alert_t = policy_config.get('alert_threshold', 0.9186015432508062)
    print(f'  Alert threshold:  {alert_t}')
    print(f'  All thresholds frozen since Phase 9/10. No adaptation allowed.')
except Exception as e:
    # Fallback to hardcoded values
    print('FROZEN DECISION POLICY (from authoritative records)')
    print('=' * 100)
    policy = pd.DataFrame({
        'Risk Level': ['ALERT', 'BORDERLINE', 'MONITOR', 'NON-ALERT'],
        'Condition': [
            'score >= 0.9186',
            '0.8686 <= score < 0.9186',
            '0.4635 <= score < 0.8686',
            'score < 0.4635'
        ],
        'Action': [
            'escalate_to_incident_response',
            'queue_for_analyst_review',
            'add_to_watchlist',
            'no_action_required'
        ],
        'Urgency': ['immediate', 'within_24h', 'weekly', 'none']
    })
    print(policy.to_string(index=False))
print()
print('NOTE: alert_flag = True for ALERT + BORDERLINE (broader than Phase 10 ALERT-only).')
print('The 58 additional BORDERLINE flags are expected per output_schema.yaml.')

## Final LightGBM Model

**FROZEN CANDIDATE** — `lgbm-graph-v1`

| Property | Value |
|---|---|
| Framework | LightGBM 4.6.0 |
| Seed | 42 |
| Best iteration | 186 |
| Features | 12 (8 behavioral + 4 graph) |
| Alert threshold | 0.9186015432508062 |
| Training time | 4.64 seconds |
| Model size | 651,047 bytes |

### Frozen Feature List

```yaml
behavioral:
  - login_count
  - after_hours_login_count
  - usb_connection_count
  - file_access_count
  - sensitive_file_access_count
  - http_activity_count
  - unique_device_count
  - unusual_access_count
graph:
  - device_consistency_score
  - rare_device_usage_count
  - file_type_consistency_score
  - rare_file_type_access_count
```

In [ ]:
if TABLE_LOADED:
    print(f'Phase 20 Decision Table: {df.shape[0]:,} rows x {df.shape[1]} columns')
    print(f'DATA_SOURCE: {DATA_SOURCE}')
    print(f'Full table hash verified: {FULL_TABLE_HASH_VERIFIED}')
    print(f'\nColumns: {list(df.columns)}')
    print(f'\nSample rows (first 5):')
    display(df.head(5))
else:
    print('FAIL: Decision table not loaded.')

In [ ]:
# Load frozen test results from phase7 freeze record
try:
    with open(os.path.join(PROJECT_ROOT, 'reports/artifacts/phase7_freeze_lgbm-graph-v1.json')) as f:
        freeze = json.load(f)
    test = freeze.get('test', {})
    
    print("FROZEN CHRONOLOGICAL TEST RESULT")
    print("=" * 55)
    print(f"  ROC-AUC:          {test.get('auc_roc', 'N/A')}")
    print(f"  PR-AUC:           {test.get('auc_pr', 'N/A')}")
    print(f"  Precision:        {test.get('precision', 'N/A')}")
    print(f"  Recall:           {test.get('recall', 'N/A')}")
    print(f"  F1:               {test.get('f1', 'N/A')}")
    print(f"  MCC:              {test.get('mcc', 'N/A')}")
    print(f"  Balanced Acc:     {test.get('balanced_accuracy', 'N/A')}")
    print(f"  Alerts:           {test.get('n_alerts', 'N/A')}")
except Exception as e:
    print(f"FROZEN VALUES (from authoritative records):")
    print(f"  ROC-AUC:          0.9391565538286849")
    print(f"  PR-AUC:           0.2677761042396373")
    print(f"  Precision:        0.2857142857142857")
    print(f"  Recall:           0.4666666666666667")
    print(f"  F1:               0.35443037974683544")
    print(f"  MCC:              0.3646390741124269")
    print(f"  Balanced Acc:     0.7329607550919026")
    print(f"  Alerts:           49")

print()
print("Bootstrap CIs (from freeze record):")
print("  ROC-AUC: 0.9387 [0.8997, 0.9747]")
print("  PR-AUC:  0.2865 [0.1404, 0.4602]")
print()
print("Top-K Precision:")
print("  P@10:  0.600")
print("  P@30:  0.367")
print("  R@50:  0.467")

In [ ]:
# Decision distribution - computed LIVE from full table
if TABLE_LOADED:
    dist = df['risk_level'].value_counts()
    total = len(df)
    
    print('FROZEN VALIDATED RESULT - Decision Distribution (LIVE from full table)')
    print('=' * 80)
    for level in ['ALERT', 'BORDERLINE', 'MONITOR', 'NON-ALERT']:
        count = dist.get(level, 0)
        pct = count / total * 100
        print(f'  {level:12s}  {count:>8,}  ({pct:5.2f}%)')
    print(f'  {"":12s}  {"-------":>8s}')
    print(f'  {"TOTAL":12s}  {total:>8,}')
    print()
    
    # Verify expected counts
    expected = {'ALERT': 3785, 'BORDERLINE': 1484, 'MONITOR': 178204, 'NON-ALERT': 317527}
    all_match = True
    for level, exp_count in expected.items():
        actual = dist.get(level, 0)
        if actual != exp_count:
            print(f'  WARNING: {level} expected {exp_count}, got {actual}')
            all_match = False
    
    alert_true = dist.get('ALERT', 0) + dist.get('BORDERLINE', 0)
    print(f'  Alert flag True (ALERT + BORDERLINE):  {alert_true:,} ({alert_true/total*100:.2f}%)')
    print(f'  Alert flag False (MONITOR + NON-ALERT): {total - alert_true:,} ({(total-alert_true)/total*100:.2f}%)')
    print()
    print(f'  DECISION_COUNT_SUM = {total:,}')
    print(f'  DECISION_DISTRIBUTION_MATCH = {"YES" if all_match else "NO"}')
    print()
    print('  NOTE: alert_flag = True for ALERT OR BORDERLINE (Phase 20 output_schema.yaml).')
    print('  The 58 BORDERLINE additions above ALERT-only are expected policy additions.')
else:
    print('FAIL: Decision table not loaded.')

In [ ]:
# Phase 19 CONFIRM results
confirm = pd.DataFrame({
    'Metric': ['ROC-AUC', 'PR-AUC', 'F1', 'Alerts', 'Precision', 'Recall', 'MCC'],
    'A0 (ML-only)': [0.778861, 0.298420, 0.340694, 72, 0.750000, 0.220408, 0.405881],
    'Family C': [0.750736, 0.297115, 0.338558, 74, 0.729730, 0.220408, 0.400326],
    'Delta': [-0.028125, -0.001305, -0.002136, 2, -0.020270, 0.000000, -0.005555],
})

print("FROZEN VALIDATED RESULT — Phase 19 CONFIRM")
print("=" * 75)
print(confirm.to_string(index=False))
print()
print("Verdict: FAIL")
print("  delta ROC-AUC = -0.0281 < -0.01 threshold")
print("  Adaptive Risk REJECTED for production")
print("  Frozen system UNCHANGED")

## Conformal Prediction

### What is Conformal Prediction?

Conformal prediction adds **uncertainty quantification** on top of the frozen classifier.

- It does NOT improve the classifier
- It provides **coverage guarantees** — "the true label is in the prediction set at least 95% of the time"
- It answers: "how confident should the analyst be?"

### Method: Mondrian Split Conformal

- Fit on CALIBRATION block only (n₁ = 323 positives, n₀ = 58,677 negatives)
- Alpha = 0.05 (95% coverage target)
- Prediction sets: {0}, {1}, {0,1}, or {}

**⚠️ Conformal output is NOT probability of correctness.** It is a frequentist coverage guarantee.

In [ ]:
# Conformal results
print("FROZEN VALIDATED RESULT — Conformal Coverage (alpha=0.05)")
print("=" * 60)
print(f"  Positive coverage (TEST):  1.000 (30/30)")
print(f"  Negative coverage (TEST):  0.988")
print(f"  Marginal coverage (TEST):  0.988")
print(f"  Wilson 90% LB (pos):       0.917")
print()
print("  Frozen thresholds:")
print(f"    t0 (negative):  0.4634739481800199")
print(f"    t1 (positive):  0.0046536002164601275")
print(f"    n1 (CAL pos):   323")
print(f"    n0 (CAL neg):   58,677")
print()
print("  Prediction set interpretation:")
print("    {0}    → model predicts BENIGN with 95% confidence")
print("    {1}    → model predicts MALICIOUS with 95% confidence")
print("    {0,1}  → model is UNCERTAIN (analyst should review)")
print("    {}     → empty set (rare, data issue)")
print()
print("  Monitor band (t0 ≤ score < threshold):")
print("    CAL:   765 rows, precision 0.061 (~11× prevalence)")
print("    TEST:  532 rows, precision 0.015 (~25× prevalence)")
print("    → Useful watchlist for analyst review")

In [ ]:
# SHAP findings from Phase 11
print('FROZEN VALIDATED RESULT - SHAP Explainability Findings')
print('=' * 60)
print()
print('  Coverage:')
print('    Direct SHAP (TreeSHAP):   21,043 rows (4.2% of 501K)')
print('    Feature-value fallback:   479,957 rows (95.8%)')
print()
print('  Alert-level feature importance (49 TEST alerts):')
print('  +------------------------------+--------------+--------------------+')
print('  | Feature                      | Top-1 Count  | Role               |')
print('  +------------------------------+--------------+--------------------+')
print('  | usb_connection_count          | 36/49 (73%)  | Dominant alert driver|')
print('  | http_activity_count           |  3/49        | Bipolar (~49% global)|')
print('  | device_consistency_score      |  9/49        | Top graph feature   |')
print('  | file_type_consistency_score   |  5/49        | Graph contributor   |')
print('  | file_access_count             |  8/49        | Secondary behavioral|')
print('  | after_hours_login_count       |  4/49        | Minor               |')
print('  | login_count                   |  2/49        | Minor               |')
print('  | unique_device_count           |  3/49        | Minor               |')
print('  | unusual_access_count          |  2/49        | Minor               |')
print('  | rare_file_type_access_count   |  1/49        | Minor               |')
print('  | sensitive_file_access_count   |  0/49        | Never top-1         |')
print('  | rare_device_usage_count       |  0/49        | Never top-1         |')
print('  +------------------------------+--------------+--------------------+')
print()
print('  Validation:')
print('    Max TreeExplainer difference: 0.0 (bit-identical)')
print('    Reconstruction error:         ~5e-14 (essentially exact)')
print()
print('  Graph feature contribution: ~20.5% of alert-row |contribution|')
print('  Stability: decision flips <= 3.5% per feature under unit perturbation')
print('  caveat: top-3 rank churn ~ 95% (reason rank is fragile)')
print()
print('  IMPORTANT: SHAP shows what features contributed to elevated model risk.')
print('  It does NOT show what caused the insider threat (non-causal).')

## SHAP Explainability

### Method
- LightGBM native `pred_contrib` (path-dependent TreeSHAP)
- Operates in **margin space**: margin = bias + Σ contribution
- Deterministic, label-free, non-causal

### Validation
- Max difference vs shap TreeExplainer: **0.0** (bit-identical)
- Reconstruction error: ~5e-14 (essentially exact)

### What SHAP Provides
- Top-3 reasons per user-day
- Feature contribution magnitudes
- Global feature importance

### What SHAP Does NOT Provide
- Causal explanations
- Counterfactual predictions
- Probability of correctness

In [ ]:
# SHAP findings from Phase 11
print("FROZEN VALIDATED RESULT — SHAP Explainability Findings")
print("=" * 60)
print()
print("  Alert-level feature importance (49 TEST alerts):")
print("  ┌─────────────────────────────────┬──────────────┬────────────────────┐")
print("  │ Feature                         │ Top-1 Count  │ Role               │")
print("  ├─────────────────────────────────┼──────────────┼────────────────────┤")
print("  │ usb_connection_count            │ 36/49 (73%)  │ Dominant alert driver│")
print("  │ http_activity_count             │  3/49        │ Bipolar (~49% global)│")
print("  │ device_consistency_score        │  9/49        │ Top graph feature   │")
print("  │ file_type_consistency_score     │  5/49        │ Graph contributor   │")
print("  │ file_access_count               │  8/49        │ Secondary behavioral│")
print("  │ after_hours_login_count         │  4/49        │ Minor               │")
print("  │ login_count                     │  2/49        │ Minor               │")
print("  │ unique_device_count             │  3/49        │ Minor               │")
print("  │ unusual_access_count            │  2/49        │ Minor               │")
print("  │ rare_file_type_access_count     │  1/49        │ Minor               │")
print("  │ sensitive_file_access_count     │  0/49        │ Never top-1         │")
print("  │ rare_device_usage_count         │  0/49        │ Never top-1         │")
print("  └─────────────────────────────────┴──────────────┴────────────────────┘")
print()
print("  Graph feature contribution: ~20.5% of alert-row |contribution|")
print("  Stability: decision flips ≤ 3.5% per feature under unit perturbation")
print("  caveat: top-3 rank churn ≈ 95% (reason rank is fragile)")

In [ ]:
# SHAP visualization
fig, ax = plt.subplots(figsize=(10, 5))

features = ['usb_connection\ncount', 'device\nconsistency', 'file_type\nconsistency', 
            'file_access\ncount', 'after_hours\nlogin', 'http\nactivity',
            'login\ncount', 'unique\ndevice', 'unusual\naccess', 'rare\nfile_type',
            'sensitive\nfile', 'rare\ndevice']
top1_counts = [36, 9, 5, 8, 4, 3, 2, 3, 2, 1, 0, 0]
colors_shap = ['#e74c3c' if c > 10 else '#f39c12' if c > 0 else '#95a5a6' for c in top1_counts]

bars = ax.barh(features, top1_counts, color=colors_shap)
ax.set_xlabel('Top-1 Count (of 49 TEST alerts)')
ax.set_title('SHAP Feature Importance: Top-1 Reason Count\n(FROZEN VALIDATED RESULT)', fontweight='bold')
ax.invert_yaxis()

for bar, count in zip(bars, top1_counts):
    if count > 0:
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, 
                str(count), va='center', fontweight='bold')

plt.subplots_adjust(hspace=0.35, wspace=0.3)
plt.show()

## Final Decision Policy

### Frozen Risk Levels

Read from `configs/phase20/decision_policy.yaml`:

In [ ]:
# Decision policy from frozen config
policy = pd.DataFrame({
    'Risk Level': ['ALERT', 'BORDERLINE', 'MONITOR', 'NON-ALERT'],
    'Condition': [
        'score >= 0.9186',
        '0.8686 <= score < 0.9186',
        '0.4635 <= score < 0.8686',
        'score < 0.4635'
    ],
    'Confidence': [0.95, 0.90, 0.90, 0.95],
    'Conformal Set': ['{1}', '{0,1}', '{0,1}', '{0}'],
    'Action': [
        'escalate_to_incident_response',
        'queue_for_analyst_review',
        'add_to_watchlist',
        'no_action_required'
    ],
    'Urgency': ['immediate', 'within_24h', 'weekly', 'none']
})

print("FROZEN DECISION POLICY — configs/phase20/decision_policy.yaml")
print("=" * 100)
print(policy.to_string(index=False))
print()
print("BORDERLINE width: 0.05 (= 0.9186 - 0.8686)")
print("All thresholds frozen since Phase 9/10. No adaptation allowed.")

In [ ]:
print('FROZEN VALIDATED RESULT - Coverage Statistics')
print('=' * 55)
print()
print('  Conformal coverage:')
print(f'    Available:        501,000 / 501,000 (100%)')
print(f'    High-confidence:  61,017 (12.2%) - ALERT or NON-ALERT')
print(f'    Ambiguous:        439,983 (87.8%) - BORDERLINE or MONITOR')
print()
print('  SHAP explanation coverage:')
print(f'    Direct SHAP:      21,043 (4.2%) - TEST-alert rows')
print(f'    Feature fallback: 479,957 (95.8%) - remaining rows')
print()
print('  Trust diagnostics:')
print(f'    With trust flags: 476,200 (95.1%)')
print(f'    Without flags:     24,800 (4.9%)')
print()
print('  All 501,000 rows have top_3_reasons populated.')
print('  All 49 TEST alerts are confident-positive conformal set {{1}}.')
print()
print('  NOTE: Trust diagnostics are analyst context only.')
print('  They must NOT alter the final ML risk score.')
print('  Adaptive Risk remains REJECTED for production.')

## Decision Distribution

In [ ]:
# Decision distribution
dist = pd.DataFrame({
    'Risk Level': ['ALERT', 'BORDERLINE', 'MONITOR', 'NON-ALERT'],
    'Count': [3785, 1484, 178204, 317527],
    'Percentage': [0.76, 0.30, 35.57, 63.38],
    'Action': ['escalate_to_incident_response', 'queue_for_analyst_review',
              'add_to_watchlist', 'no_action_required'],
    'Urgency': ['immediate', 'within_24h', 'weekly', 'none']
})

print("FROZEN VALIDATED RESULT — Decision Distribution (501,000 user-days)")
print("=" * 80)
print(dist.to_string(index=False))
print()
print(f"  Alert flag True (ALERT + BORDERLINE):  {3785+1484:,} ({(3785+1484)/501000*100:.2f}%)")
print(f"  Alert flag False (MONITOR + NON-ALERT): {178204+317527:,} ({(178204+317527)/501000*100:.2f}%)")
print()
print("This is the OPERATIONAL DISTRIBUTION, not new classifier-performance evidence.")

In [ ]:
# Decision distribution chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

levels = ['ALERT', 'BORDERLINE', 'MONITOR', 'NON-ALERT']
counts = [3785, 1484, 178204, 317527]
colors_dist = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']

axes[0].bar(levels, counts, color=colors_dist)
axes[0].set_title('Decision Distribution (501K user-days)')
axes[0].set_ylabel('Count')
for i, (l, c) in enumerate(zip(levels, counts)):
    axes[0].text(i, c + 5000, f'{c:,}', ha='center', fontsize=9, fontweight='bold')

axes[1].pie(counts, labels=levels, colors=colors_dist, autopct='%1.2f%%', startangle=90)
axes[1].set_title('Risk Level Proportions')

plt.suptitle('Phase 20 Operational Decision Distribution (FROZEN)', fontweight='bold')
plt.subplots_adjust(hspace=0.35, wspace=0.3)
plt.show()

## Confidence / Coverage

In [ ]:
def show_user_day(user_id, date):
    """Display the decision for a specific user-day from the Phase 20 table."""
    if not TABLE_LOADED:
        print('FAIL: Decision table not loaded. Cannot display user-day.')
        return
    
    # Adaptive column detection
    uid_col = next((c for c in ['user_id','user'] if c in df.columns), 'user_id')
    date_col = next((c for c in ['date','day'] if c in df.columns), 'date')
    risk_col = next((c for c in ['ml_risk_score','ml_risk'] if c in df.columns), 'ml_risk_score')
    conf_set_col = next((c for c in ['conformal_prediction_set','conformal_set'] if c in df.columns), 'conformal_prediction_set')
    trust_col = next((c for c in ['trust_diagnostic_summary','trust_flags','trust_flag'] if c in df.columns), 'trust_diagnostic_summary')
    if 'top_reason_1' in df.columns:
        reason_cols = ('top_reason_1', 'top_reason_2', 'top_reason_3')
    elif 'top_1_reason' in df.columns:
        reason_cols = ('top_1_reason', 'top_2_reason', 'top_3_reason')
    elif 'top_reasons' in df.columns:
        reason_cols = ('top_reasons', None, None)
    else:
        reason_cols = (None, None, None)
    
    row = df[(df[uid_col] == user_id) & (df[date_col] == date)]
    if row.empty:
        print(f'No record found for user={user_id}, date={date}')
        return
    
    row = row.iloc[0]
    
    print('=' * 60)
    print(f'  User-Day Decision: {user_id} on {date}')
    print('=' * 60)
    print(f'  Risk Score:        {row[risk_col]:.6f}')
    print(f'  Risk Level:        {row["risk_level"]}')
    print(f'  Confidence:        {row["confidence"]}')
    conf_status = row.get('confidence_status', None)
    if conf_status is not None:
        print(f'  Confidence Status: {conf_status}')
    print(f'  Conformal Set:     {row[conf_set_col]}')
    p1 = row.get('conformal_p1', None)
    p0 = row.get('conformal_p0', None)
    if p1 is not None:
        print(f'  Conformal p1:      {p1:.6f}')
    if p0 is not None:
        print(f'  Conformal p0:      {p0:.6f}')
    print(f'  Alert Flag:        {row["alert_flag"]}')
    print(f'  Recommended:       {row["recommended_action"]}')
    print(f'  Urgency:           {row["urgency"]}')
    print(f'  Rationale:         {row["rationale"]}')
    trust_val = row.get(trust_col, None)
    if trust_val is not None:
        print(f'  Trust Diagnostics: {trust_val}')
    else:
        print(f'  Trust Diagnostics: Not available in frozen output')
    n_flags = row.get('n_trust_flags', None)
    if n_flags is not None:
        print(f'  Trust Flag Count:  {n_flags}')
    if reason_cols[0] and reason_cols[0] in df.columns:
        print(f'  Top Reason 1:      {row[reason_cols[0]]}')
        if reason_cols[1] and reason_cols[1] in df.columns and reason_cols[1] != reason_cols[0]:
            print(f'  Top Reason 2:      {row[reason_cols[1]]}')
        if reason_cols[2] and reason_cols[2] in df.columns and reason_cols[2] != reason_cols[0]:
            print(f'  Top Reason 3:      {row[reason_cols[2]]}')
    else:
        print(f'  Top Reason 1:      Not available in frozen output')
    model_v = row.get('model_version', None)
    if model_v is not None:
        print(f'  Model Version:     {model_v}')
    policy_v = row.get('policy_version', None)
    if policy_v is not None:
        print(f'  Policy Version:    {policy_v}')
    explanation_v = row.get('explanation_version', None)
    if explanation_v is not None:
        print(f'  Explanation Ver:   {explanation_v}')
    print('=' * 60)

# Demo: show examples from all four risk levels (REAL rows from the table)
if TABLE_LOADED:
    uid_col = next((c for c in ['user_id','user'] if c in df.columns), 'user_id')
    date_col = next((c for c in ['date','day'] if c in df.columns), 'date')
    for level in ['ALERT', 'BORDERLINE', 'MONITOR', 'NON-ALERT']:
        sample = df[df['risk_level'] == level].head(1)
        if not sample.empty:
            s = sample.iloc[0]
            show_user_day(s[uid_col], s[date_col])
            print()
else:
    print('FAIL: Cannot display user-day examples. Table not loaded.')


## Operational Validation

### Score Integrity

Every prediction was cross-checked against the Phase 7 frozen predictions.

In [ ]:
print("FROZEN VALIDATED RESULT — Operational Validation")
print("=" * 55)
print()
print("  Score Integrity:")
print(f"    Comparisons:       47,000 (TEST overlap)")
print(f"    Mismatches:        0")
print(f"    Max abs diff:      5.55e-17")
print()
print("  Alert Integrity:")
print(f"    Total differences: 58")
print(f"    ALERT-only mismatches: 0")
print(f"    BORDERLINE additions:  58 (expected per output_schema.yaml)")
print()
print("  Determinism:")
print(f"    Run A MD5:  ad017e59168a2a211d0ba74422132d46")
print(f"    Run B MD5:  ad017e59168a2a211d0ba74422132d46")
print(f"    Verdict:    PASS")
print()
print("  Test Suite:")
print(f"    Total:      54")
print(f"    Passed:     54")
print(f"    Failed:     0")
print()
print("  Performance:")
print(f"    Runtime:    180.34 seconds")
print(f"    Peak mem:   1,744.2 MB")
print()
print("  ✅ All validation gates PASS")

## Full A–F Ablation (End-to-End)

### ⚠️ Important: Direct-Comparability Distinction

| Entries | Cohort | Comparable? |
|---|---|---|
| A / B / C / D1 | Chronological TEST (47K rows) | ✅ Yes, directly comparable |
| D2 | Phase 19 CONFIRM (98K rows) | ❌ Different cohort |
| E / F | System-layer evidence | ❌ Not classifier-ranking gains |

## Final Conclusion

### What Was Built

A **leakage-resistant insider threat detection system** for CERT r4.2:

1. **Behavioral features** provided the strong baseline
2. **Graph features** improved the combined detector (+74.9% PR-AUC)
3. **Adaptive Risk** failed validation and was removed (scientific integrity)
4. **Conformal prediction** added honest uncertainty quantification
5. **SHAP** added interpretable explanations
6. **Phase 20** converted the detector into an operational decision system
7. **Reproducibility** and leakage controls were preserved throughout

### Final Production Architecture

```
Behavioral + Graph Features (12)
-> LightGBM lgbm-graph-v1 (frozen)
-> Frozen Policy (threshold 0.9186)
-> Conformal Uncertainty (alpha=0.05)
-> SHAP Explanations
-> Trust Diagnostics
-> Decision Engine (501K x 21)
```

## How We Prevented Leakage

Leakage prevention was a core design principle throughout the project.

| Control | Status |
|---|---|
| Chronological split (no random splitting) | ✅ Enforced |
| Separate calibration block | ✅ CAL isolated from TRAIN and TEST |
| No threshold tuning on TEST | ✅ Thresholds from CAL only |
| Graph statistics computed leakage-safely | ✅ Strictly-past first-use rule |
| No future information in features | ✅ Day-local or 28-day strictly-past |
| PH19_CONFIRM used once | ✅ One-time evaluation, rerun forbidden |
| Threshold leakage discovered and corrected | ✅ Before scientific CONFIRM |
| Production table contains no label | ✅ is_malicious = evaluation only |
| No post-hoc Adaptive Risk rescue | ✅ Adaptive Risk rejected and excluded |
| TEST evaluated exactly once | ✅ Per finalized experiment |
| Conformal fit on CAL only | ✅ No TEST data in calibration |
| Explanations are label-free | ✅ Selection by scores only |

## Questions I Expect From My Teacher

### Q1: Why user x day?
Insider threats manifest as daily behavioral anomalies. User-day granularity prevents label leakage from adjacent days, enables chronological splitting, and matches analyst workflow (daily review).

### Q2: Why not random split?
Behavioral data is temporally autocorrelated. Random splitting would leak future information into training, producing optimistically biased results. Chronological splitting mimics real deployment.

### Q3: Why LightGBM?
LightGBM is fast, handles class imbalance well (via scale_pos_weight), produces interpretable feature importances, and was the best performer in controlled comparison. It trains in ~5 seconds on this dataset.

### Q4: Why graph features?
Graph features capture relational patterns (device consistency, file-type consistency) that behavioral counts alone miss. They improved PR-AUC by 74.9% relative.

### Q5: Why did graph-only perform poorly?
Graph features are sparse (~99.9% zero for rare counts) and collapsed the model to a stump (3 iterations). They are useful only as complementary signals, not standalone.

### Q6: Why reject Adaptive Risk?
Three independent attempts (Phase 8, 18, 19) all failed. The learned weights collapsed to ML-only [1,0,0]. Phase 19 CONFIRM showed delta ROC-AUC -0.0281 < -0.01 threshold. Scientific integrity demanded rejection.

### Q7: What does conformal prediction mean?
Conformal prediction adds uncertainty quantification. At alpha=0.05, it guarantees that the true label is in the prediction set for at least 95% of future samples (under exchangeability). It does NOT improve the classifier.

### Q8: Why is conformal not probability?
Conformal sets are frequentist coverage objects, not probability estimates. A 95% conformal set means the coverage rate is >=95% over many repetitions. It does NOT mean that any individual prediction has 95% probability of being correct.

### Q9: Why SHAP?
SHAP provides local feature contributions for each prediction, enabling analyst understanding. It is deterministic, label-free, and validated against TreeExplainer (max diff = 0.0).

### Q10: Is SHAP causal?
No. SHAP shows what features contributed to the model's risk score, not what caused the insider threat. The notebook uses 'contributed to elevated model risk' language.

### Q11: Why only 4.2% direct SHAP in full table?
Direct SHAP (TreeSHAP) is computationally expensive. For the 501K full table, feature-value fallback was used for non-alert rows. All 49 TEST alerts have direct SHAP explanations.

### Q12: How do you prevent leakage?
Chronological splits, separate calibration, strictly-past windows, no TEST tuning, label-free explanations, frozen thresholds, and documented controls throughout.

### Q13: Why PR-AUC over accuracy?
With 0.38% prevalence, accuracy is meaningless (99.62% by predicting all-benign). PR-AUC measures discriminative power under the actual class distribution.

### Q14: Why 49 frozen TEST alerts but 3,785 full-table ALERT rows?
The 49 alerts are from the chronological TEST set only (47K rows, 30 positives). The 3,785 ALERT rows are from the full 501K table spanning the entire date range.

### Q15: What is BORDERLINE?
BORDERLINE is a risk level for scores within 0.05 below the alert threshold. It triggers analyst review within 24 hours, providing a buffer zone for near-threshold cases.

### Q16: Why not GNN / transformer?
These were out of scope per project principles. The project focused on a production-style system with explainable, reproducible components. GNNs/transformers would require different infrastructure.

### Q17: Can PH19_CONFIRM be rerun?
No. The state is confirm_opened=true, confirm_completed=true. Rerun is forbidden per spec. This ensures the evaluation remains a one-time, unbiased test.

### Q18: What are the biggest limitations?
30 TEST positives (wide uncertainty), user-entity transport degradation (AUC-ROC -0.153), tail-window temporal degradation, and CERT being a synthetic dataset.

### Q19: How would this work in a real organization?
The architecture (frozen model -> conformal -> SHAP -> decision engine) is production-ready. A real deployment would need: real-time feature computation, dashboard integration, analyst feedback loops, and periodic recalibration.

### Q20: What is your final contribution?
A complete, reproducible, leakage-resistant insider threat detection system with: 12-feature LightGBM classifier, conformal uncertainty, SHAP explanations, trust diagnostics, and a deterministic 501K-row decision table -- all validated through 20 phases of controlled experimentation with documented rejections of failed approaches.

## Final System Demo

### `show_user_day(user_id, date)` — Operational Display

## Final Conclusion

### What Was Built

A **leakage-resistant insider threat detection system** for CERT r4.2:

1. **Behavioral features** provided the strong baseline
2. **Graph features** improved the combined detector (+74.9% PR-AUC)
3. **Adaptive Risk** failed validation and was removed (scientific integrity)
4. **Conformal prediction** added honest uncertainty quantification
5. **SHAP** added interpretable explanations
6. **Phase 20** converted the detector into an operational decision system
7. **Reproducibility** and leakage controls were preserved throughout

### Final Production Architecture

```
Behavioral + Graph Features (12)
→ LightGBM lgbm-graph-v1 (frozen)
→ Frozen Policy (threshold 0.9186)
→ Conformal Uncertainty (alpha=0.05)
→ SHAP Explanations
→ Trust Diagnostics
→ Decision Engine (501K × 21)
```

## Limitations

| Category | Limitation |
|---|---|
| Data | Extreme imbalance (0.38% prevalence) |
| Data | Only 30 chronological TEST positives |
| Evaluation | Wide PR-AUC confidence interval [0.140, 0.460] |
| Generalization | User/entity transport degradation (AUC-ROC 0.786 vs 0.939) |
| Temporal | Tail-window degradation documented (Phase 13 W8) |
| Conformal | Exchangeability assumption (no distribution-shift guarantee) |
| Explainability | SHAP ≠ causality |
| Coverage | Direct SHAP only 4.2% of full table |
| Coverage | Trust diagnostics 95.1% (not 100%) |
| External validity | CERT is a synthetic dataset |
| Historical | Kaggle artifact loss (Phase 18) |

## Questions I Expect From My Teacher

### Q1: Why user × day?
Insider threats manifest as daily behavioral anomalies. User-day granularity prevents label leakage from adjacent days, enables chronological splitting, and matches analyst workflow (daily review).

### Q2: Why not random split?
Behavioral data is temporally autocorrelated. Random splitting would leak future information into training, producing optimistically biased results. Chronological splitting mimics real deployment.

### Q3: Why LightGBM?
LightGBM is fast, handles class imbalance well (via scale_pos_weight), produces interpretable feature importances, and was the best performer in controlled comparison. It trains in ~5 seconds on this dataset.

### Q4: Why graph features?
Graph features capture relational patterns (device consistency, file-type consistency) that behavioral counts alone miss. They improved PR-AUC by 74.9% relative.

### Q5: Why did graph-only perform poorly?
Graph features are sparse (~99.9% zero for rare counts) and collapsed the model to a stump (3 iterations). They are useful only as complementary signals, not standalone.

### Q6: Why reject Adaptive Risk?
Three independent attempts (Phase 8, 18, 19) all failed. The learned weights collapsed to ML-only [1,0,0]. Phase 19 CONFIRM showed delta ROC-AUC -0.0281 < -0.01 threshold. Scientific integrity demanded rejection.

### Q7: What does conformal prediction mean?
Conformal prediction adds uncertainty quantification. It guarantees "the true label is in the prediction set at least 95% of the time" (under exchangeability). It does NOT improve the classifier.

### Q8: Why is conformal not probability?
Conformal sets are frequentist coverage objects, not probability estimates. A 95% conformal set means the coverage rate is ≥95% over many repetitions, not that any individual prediction has 95% probability of being correct.

### Q9: Why SHAP?
SHAP provides local feature contributions for each prediction, enabling analyst understanding. It is deterministic, label-free, and validated against TreeExplainer (max diff = 0.0).

### Q10: Is SHAP causal?
No. SHAP shows what features contributed to the model's risk score, not what caused the insider threat. The notebook uses "contributed to elevated model risk" language.

### Q11: Why only 4.2% direct SHAP in full table?
Direct SHAP (TreeSHAP) is computationally expensive. For the 501K full table, feature-value fallback was used for non-alert rows. All 49 TEST alerts have direct SHAP explanations.

### Q12: How do you prevent leakage?
Chronological splits, separate calibration, strictly-past windows, no TEST tuning, label-free explanations, frozen thresholds, and documented controls throughout.

### Q13: Why PR-AUC over accuracy?
With 0.38% prevalence, accuracy is meaningless (99.62% by predicting all-benign). PR-AUC measures discriminative power under the actual class distribution.

### Q14: Why 49 frozen TEST alerts but 3,785 full-table ALERT rows?
The 49 alerts are from the chronological TEST set only (47K rows, 30 positives). The 3,785 ALERT rows are from the full 501K table spanning the entire date range.

### Q15: What is BORDERLINE?
BORDERLINE is a risk level for scores within 0.05 below the alert threshold. It triggers analyst review within 24 hours, providing a buffer zone for near-threshold cases.

### Q16: Why not GNN / transformer?
These were out of scope per project principles. The project focused on a production-style system with explainable, reproducible components. GNNs/transformers would require different infrastructure.

### Q17: Can PH19_CONFIRM be rerun?
No. The state is `confirm_opened=true, confirm_completed=true`. Rerun is forbidden per spec. This ensures the evaluation remains a one-time, unbiased test.

### Q18: What are the biggest limitations?
30 TEST positives (wide uncertainty), user-entity transport degradation (AUC-ROC -0.153), tail-window temporal degradation, and CERT being a synthetic dataset.

### Q19: How would this work in a real organization?
The architecture (frozen model → conformal → SHAP → decision engine) is production-ready. A real deployment would need: real-time feature computation, dashboard integration, analyst feedback loops, and periodic recalibration.

### Q20: What is your final contribution?
A complete, reproducible, leakage-resistant insider threat detection system with: 12-feature LightGBM classifier, conformal uncertainty, SHAP explanations, trust diagnostics, and a deterministic 501K-row decision table — all validated through 20 phases of controlled experimentation with documented rejections of failed approaches.